In [1]:
from dataclasses import dataclass
from faster_whisper.transcribe import Segment

@dataclass
class TranscriptionResult:
    text: str
    language: str
    language_probability: float
    segments: list[Segment]
    audio_duration_s: float
    inference_time_s: float

    @property
    def realtime_factor(self) -> float:
        return self.audio_duration_s / self.inference_time_s

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from faster_whisper import WhisperModel

device = "cpu"
compute_type = "float32"
model = WhisperModel("large-v3-turbo", device=device, compute_type=compute_type)

In [ ]:
import time

def transcribe(model: WhisperModel, audio_path: str, language: str | None = None, beam_size: int = 5) -> TranscriptionResult:
    """
    Transcribe an audio file.

    Args:
        model: Loaded WhisperModel instance.
        audio_path: Path to audio file (wav, mp3, m4a, etc.).
        language: ISO 639-1 code e.g. "id" for Indonesian, "en" for English.
                  None = auto-detect.
        beam_size: Beam search width. Higher = more accurate, slower.

    Returns:
        TranscriptionResult with text, segments, timing info.
    """

    t0 = time.perf_counter()
    
    segments_gen, info = model.transcribe(
        audio_path,
        language=language,
        beam_size=beam_size,
        vad_filter=True,
        vad_parameters=dict(
            min_silence_duration_ms=2000,  # wait 2s of silence before cutting
            speech_pad_ms=400,             # pad speech edges to avoid clipping
            threshold=0.3,                 # lower = more sensitive to speech
        ),
        word_timestamps=True
    )

    segments = list(segments_gen)
    full_text = " ".join(seg.text.strip() for seg in segments)
    elapsed = time.perf_counter() - t0

    return TranscriptionResult(
        text=full_text,
        language=info.language,
        language_probability=info.language_probability,
        segments=segments,
        audio_duration_s=info.duration,
        inference_time_s=elapsed
    )

In [4]:
import tempfile
import sounddevice as sd
import soundfile as sf

# record audio for testing
duration_s = 10
sample_rate = 16_000
print("Starting recording...")
audio = sd.rec(
    int(duration_s * sample_rate),
    samplerate=sample_rate,
    channels=1,
    dtype="float32"
)

sd.wait()
print("Done recording")

with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
    sf.write(f.name, audio, sample_rate)
    result = transcribe(model, f.name)

Starting recording...
Done recording


In [5]:
print(f"Language : {result.language} ({result.language_probability:.0%})")
print(f"Duration : {result.audio_duration_s:.1f}s")
print(f"Inference: {result.inference_time_s:.2f}s ({result.realtime_factor:.1f}x realtime)")
print(f"Transcript:\n{result.text}")

Language : en (99%)
Duration : 10.0s
Inference: 4.31s (2.3x realtime)
Transcript:
I have a question. Yes! I'm in the mood for lamb, but the sauce looks too fatty.
